# WolBanking77 — version Colab simplifiée

Ce notebook entraîne et compare deux modèles pour reconnaître les intentions bancaires en wolof. Exécutez les cellules dans l'ordre (ou **Exécution > Exécuter tout**).

**Avant de commencer :** dans Colab, choisissez `Exécution > Modifier le type d'exécution > T4 GPU` pour entraîner XLM-RoBERTa plus vite.

## 1. Connecter Google Drive et récupérer le projet

Les modèles et rapports seront enregistrés dans Google Drive. Les données sont récupérées depuis le dépôt GitHub.

In [ ]:
from google.colab import drive
from pathlib import Path

REPO_URL = 'https://github.com/alimar440/Projet-WolBanking77.git'
PROJECT_DIR = Path('/content/WolBanking77')
OUTPUT_DIR = Path('/content/drive/MyDrive/WolBanking77_runs')

drive.mount('/content/drive')
if not PROJECT_DIR.exists():
    !git clone $REPO_URL $PROJECT_DIR
else:
    %cd $PROJECT_DIR
    !git pull
%cd $PROJECT_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Projet :', PROJECT_DIR)
print('Résultats :', OUTPUT_DIR)

## 2. Installer les bibliothèques et choisir les paramètres

`DATASET_NAME = '5k_split'` est le jeu rapide (4 000 exemples train / 1 000 test). Remplacez-le par `full` pour le jeu complet.

In [ ]:
!pip -q install transformers sentencepiece joblib

import json, random, re
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

# Paramètres d'entraînement identiques à WolBanking77_Colab_V3.ipynb
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DATASET_NAME = '5k_split'  # ou 'full'
MODEL_NAME = 'xlm-roberta-base'
MAX_LENGTH = 64
BATCH_SIZE = 16 if torch.cuda.is_available() else 4
LEARNING_RATE, EPOCHS, PATIENCE = 2e-5, 20, 3
HEAD_LR_MULTIPLIER = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_DIR = PROJECT_DIR / 'data' / DATASET_NAME
MODELS_DIR = OUTPUT_DIR / 'models'
REPORTS_DIR = OUTPUT_DIR / 'reports'
MODELS_DIR.mkdir(exist_ok=True); REPORTS_DIR.mkdir(exist_ok=True)
print('Appareil :', DEVICE)
print('Données :', DATA_DIR)

## 3. Charger et vérifier les données

In [ ]:
def clean_text(text):
    return re.sub(r'\s+', ' ', str(text).strip())

def read_split(name):
    frame = pd.read_csv(DATA_DIR / name / f'{name}.csv')
    frame = frame.dropna(subset=['input_wo', 'label']).copy()
    frame['text'] = frame['input_wo'].map(clean_text)
    return frame[frame['text'].ne('')].drop_duplicates('text').reset_index(drop=True)

train_df = read_split('train')
test_df = read_split('test')
label_encoder = LabelEncoder().fit(train_df['label'])
train_df['label_id'] = label_encoder.transform(train_df['label'])
test_df['label_id'] = label_encoder.transform(test_df['label'])

print(f'Train : {len(train_df)} exemples')
print(f'Test  : {len(test_df)} exemples')
print(f'Intentions : {len(label_encoder.classes_)}')
display(train_df[['input_wo', 'label']].head())

## 4. Baseline rapide : TF-IDF + LinearSVC

Ce modèle sert de référence. Il est rapide à entraîner et permet de savoir si le Transformer apporte un gain.

In [ ]:
baseline = Pipeline([
    ('features', FeatureUnion([
        ('mots', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_features=60_000)),
        ('caracteres', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True, max_features=80_000)),
    ])),
    ('classifier', LinearSVC(C=1.0))
])

baseline.fit(train_df['text'], train_df['label'])
baseline_predictions = baseline.predict(test_df['text'])
baseline_f1 = f1_score(test_df['label'], baseline_predictions, average='macro', zero_division=0)
joblib.dump(baseline, MODELS_DIR / f'baseline_{DATASET_NAME}.joblib')
print(f'Macro-F1 baseline : {baseline_f1:.2%}')

## 5. Préparer XLM-RoBERTa

Cette cellule prépare les textes pour le Transformer et réserve 10 % du train pour la validation.

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, frame, tokenizer):
        self.texts = frame['text'].tolist()
        self.labels = frame['label_id'].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        tokens = self.tokenizer(self.texts[index], truncation=True, max_length=MAX_LENGTH,
                                padding='max_length', return_tensors='pt')
        return {'input_ids': tokens['input_ids'].flatten(),
                'attention_mask': tokens['attention_mask'].flatten(),
                'labels': torch.tensor(self.labels[index], dtype=torch.long)}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_part, val_part = train_test_split(train_df, test_size=0.10, stratify=train_df['label_id'], random_state=SEED)

def make_loader(frame, shuffle=False):
    return DataLoader(IntentDataset(frame, tokenizer), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=2, pin_memory=torch.cuda.is_available())

train_loader = make_loader(train_part, shuffle=True)
val_loader = make_loader(val_part)
test_loader = make_loader(test_df)

## 6. Entraîner le Transformer

Le meilleur modèle de validation est automatiquement sauvegardé dans Google Drive.

In [ ]:
def run_epoch(model, loader, optimizer=None, scheduler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    losses, predictions, labels = [], [], []
    for batch in loader:
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        if training: optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            output = model(**batch)
            if training:
                output.loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step()
        losses.append(output.loss.item())
        predictions.extend(output.logits.argmax(1).detach().cpu().tolist())
        labels.extend(batch['labels'].detach().cpu().tolist())
    return np.mean(losses), f1_score(labels, predictions, average='macro', zero_division=0), predictions

id2label = dict(enumerate(label_encoder.classes_))
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(id2label), id2label=id2label,
    label2id={label: index for index, label in id2label.items()}
).to(DEVICE)
head_params = [p for name, p in model.named_parameters() if name.startswith('classifier')]
backbone_params = [p for name, p in model.named_parameters() if not name.startswith('classifier')]
optimizer = AdamW([
    {'params': backbone_params, 'lr': LEARNING_RATE},
    {'params': head_params, 'lr': LEARNING_RATE * HEAD_LR_MULTIPLIER},
])
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

best_f1, stale, best_dir = -1, 0, MODELS_DIR / f'xlmr_{DATASET_NAME}'
history = {'train_f1': [], 'val_f1': []}
for epoch in range(EPOCHS):
    _, train_f1, _ = run_epoch(model, train_loader, optimizer, scheduler)
    _, val_f1, _ = run_epoch(model, val_loader)
    history['train_f1'].append(train_f1); history['val_f1'].append(val_f1)
    print(f'Époque {epoch + 1}/{EPOCHS} — train F1: {train_f1:.2%} | validation F1: {val_f1:.2%}')
    if val_f1 > best_f1:
        best_f1, stale = val_f1, 0
        best_dir.mkdir(exist_ok=True)
        model.save_pretrained(best_dir)
        tokenizer.save_pretrained(best_dir)
        with open(best_dir / 'labels.json', 'w', encoding='utf-8') as file:
            json.dump(label_encoder.classes_.tolist(), file, ensure_ascii=False, indent=2)
    else:
        stale += 1
        if stale >= PATIENCE:
            print(f'Arrêt anticipé : pas d’amélioration depuis {PATIENCE} époques.')
            break

plt.plot(history['train_f1'], label='train')
plt.plot(history['val_f1'], label='validation')
plt.title('Macro-F1 pendant l’entraînement'); plt.xlabel('Époque'); plt.ylim(0, 1); plt.legend(); plt.show()

## 7. Évaluer et comparer les modèles

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(best_dir).to(DEVICE)
_, transformer_f1, prediction_ids = run_epoch(model, test_loader)
transformer_predictions = label_encoder.inverse_transform(prediction_ids)

results = pd.DataFrame({
    'Modèle': ['TF-IDF + LinearSVC', 'XLM-RoBERTa'],
    'Accuracy': [accuracy_score(test_df['label'], baseline_predictions), accuracy_score(test_df['label'], transformer_predictions)],
    'Macro-F1': [baseline_f1, transformer_f1],
})
display(results.style.format({'Accuracy': '{:.2%}', 'Macro-F1': '{:.2%}'}))

report = classification_report(test_df['label'], transformer_predictions, output_dict=True, zero_division=0)
with open(REPORTS_DIR / f'transformer_metrics_{DATASET_NAME}.json', 'w', encoding='utf-8') as file:
    json.dump(results.to_dict('records'), file, ensure_ascii=False, indent=2)
pd.DataFrame(report).T.to_csv(REPORTS_DIR / f'transformer_report_{DATASET_NAME}.csv')
print('Rapports enregistrés dans :', REPORTS_DIR)

## Test statistique global sur le jeu de test

In [ ]:
from scipy.stats import binomtest

def statistical_comparison(y_true, preds_a, preds_b, name_a, name_b, n_bootstrap=2000, seed=SEED):
    y_true = np.asarray(y_true)
    preds_a = np.asarray(preds_a)
    preds_b = np.asarray(preds_b)

    correct_a = (preds_a == y_true)
    correct_b = (preds_b == y_true)

    # Test de McNemar (exact, via test binomial sur les cas discordants)
    only_a_correct = int(np.sum(correct_a & ~correct_b))
    only_b_correct = int(np.sum(correct_b & ~correct_a))
    n_discordant = only_a_correct + only_b_correct
    mcnemar_p = binomtest(min(only_a_correct, only_b_correct), n_discordant, p=0.5).pvalue if n_discordant > 0 else 1.0

    # Intervalle de confiance bootstrap sur l'ecart de macro-F1
    rng = np.random.default_rng(seed)
    n = len(y_true)
    diffs = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        f1_a = f1_score(y_true[idx], preds_a[idx], average='macro', zero_division=0)
        f1_b = f1_score(y_true[idx], preds_b[idx], average='macro', zero_division=0)
        diffs[i] = f1_b - f1_a
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])

    print(f"{name_a} correct seul : {only_a_correct} exemples | {name_b} correct seul : {only_b_correct} exemples")
    print(f"Test de McNemar : p-value = {mcnemar_p:.4f}")
    print(f"Bootstrap 95% CI pour (macro-F1 {name_b} - macro-F1 {name_a}) : [{ci_low:.4f}, {ci_high:.4f}]")
    verdict = "significative" if (mcnemar_p < 0.05 or ci_low > 0 or ci_high < 0) else "NON significative"
    print(f"=> Difference {verdict} (seuil p < 0.05 / IC excluant 0).")

    result = {
        'mcnemar_p_value': float(mcnemar_p),
        'bootstrap_ci_macro_f1_diff': (float(ci_low), float(ci_high)),
        f'{name_a}_only_correct': only_a_correct,
        f'{name_b}_only_correct': only_b_correct,
    }
    with open(REPORTS_DIR / f'stat_test_{DATASET_NAME}.json', 'w', encoding='utf-8') as file:
        json.dump(result, file, ensure_ascii=False, indent=2)
    return result

stat_result = statistical_comparison(test_df['label'], baseline_predictions, transformer_predictions, 'baseline', 'transformer')

## 8. Tester une phrase

Modifiez la phrase ci-dessous puis relancez la cellule.

In [ ]:
def predict_intent(text):
    model.eval()
    tokens = tokenizer(clean_text(text), truncation=True, max_length=MAX_LENGTH,
                       padding='max_length', return_tensors='pt')
    with torch.no_grad():
        logits = model(input_ids=tokens['input_ids'].to(DEVICE),
                       attention_mask=tokens['attention_mask'].to(DEVICE)).logits
    probabilities = torch.softmax(logits, dim=1)[0]
    label_id = int(probabilities.argmax())
    return {'intention': label_encoder.inverse_transform([label_id])[0],
            'confiance': f'{float(probabilities[label_id]):.2%}'}

predict_intent('Xamuma lutax ñu bañ sama payoor?')